# Data Integration and Aggregation
Project : Azure E-Commerce data Pipeline

Done by : Saqib Souran

<em>I used Hardcoded credentials as I myself is working on this project. But the original credentials have been replaced by random letters </em>

---

Azure ADLS Connection.

In [0]:
# Configuration variables
storage_account = "MyADLSStorage"
application_id = "dadadq34-dad34d-dd3-fas3-fgsdkfu45464"
directory_id = "brthrtg-brtbrttrg-brtbrtbrt-bhrtr-11hrthrtgrt"
service_credential_key = "dugb8~gieuar32eryfgghgegersg4423gjk4g2b42g4"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_credential_key)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

In [0]:
# Read CSV file from ADLS Gen2 using Spark DataFrame
storage_account = "MyADLSStorage"
container_name = "olistdata"

adls_silver_path =  f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/silver/"
adls_gold_path =  f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/gold/"

customers_df = spark.read.parquet(adls_silver_path + "customers_cleaned/")
geolocation_df = spark.read.parquet(adls_silver_path + "geolocation_cleaned/")
order_item_df = spark.read.parquet(adls_silver_path + "items_cleaned")
orders_df = spark.read.parquet(adls_silver_path + "orders_cleaned/")
payments_df = spark.read.parquet(adls_silver_path + "payments_cleaned/")
products_df = spark.read.parquet(adls_silver_path + "products_cleaned/")
reviews_df = spark.read.parquet(adls_silver_path + "reviews_cleaned/")
sellers_df = spark.read.parquet(adls_silver_path + "sellers_cleaned/")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

In [0]:
def check_unique(df, keys, name):
    duplicates = (
        df.groupBy(*keys)
          .count()
          .filter(col("count") > 1)
          .count()
    )
    assert duplicates == 0, (
        f"{name} contains {duplicates} duplicate keys"
    )

check_unique(orders_df, ["order_id"], "orders")
check_unique(
    order_item_df,
    ["order_id", "order_item_id"],
    "order_items"
)
check_unique(customers_df, ["customer_id"], "customers")
check_unique(products_df, ["product_id"], "products")
check_unique(sellers_df, ["seller_id"], "sellers")

### Dimensional Tables

In [0]:
customers_df.show(10)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|cb96e80748675729d...|67626c49069bdf05f...|                   88348|            camboriu|            SC|
|279a316b31aac761c...|7fe108cef30f4cca2...|                    4003|           sao paulo|            SP|
|1914576a74b2122c6...|8574faf241b0a789d...|                    5792|           sao paulo|            SP|
|de27e7ae8adefd14b...|a50381d8ac7b457ce...|                    1224|           sao paulo|            SP|
|ad417eed258291b62...|a7b3ee56a66877702...|                    9361|                maua|            SP|
|e1a9260002f958ad5...|4965a4e8d51667a93...|                   20530|      rio de janeiro|            RJ|
|645e4bcb4b050ff31...|126646ff7e983d825...|            

In [0]:
dim_customer = customers_df

display(dim_customer.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
cb96e80748675729d4d3857321cacb26,67626c49069bdf05ff127d59a6649948,88348,camboriu,SC
279a316b31aac761c16483859fa9d33b,7fe108cef30f4cca2f8696a711a07d0b,4003,sao paulo,SP
1914576a74b2122c63a456293cf7e52c,8574faf241b0a789d42051aa783d46dd,5792,sao paulo,SP
de27e7ae8adefd14b55e07db50c5c037,a50381d8ac7b457ced2071af9ef0caed,1224,sao paulo,SP
ad417eed258291b622014bf443a0633b,a7b3ee56a66877702d89d6e6d8ecfa59,9361,maua,SP
e1a9260002f958ad5765a6e85172e0ae,4965a4e8d51667a93ac39e04aefb33be,20530,rio de janeiro,RJ
645e4bcb4b050ff31253516e363bf301,126646ff7e983d825b46a004555996bc,89694,faxinal dos guedes,SC
5833a1a8d62b965ca0f8205a8bcb32fa,16df396804bc71ff9a59764e3c0f490c,51335,recife,PE
cb1d1ff0489e8f800375a08a8b8fd92b,33f86281d5203b278e32070d4a02592e,36900,manhuacu,MG
f0244902c07b7c0799e468218579a925,147375750bdc81b55b75dc08daf84c8a,55022,caruaru,PE


In [0]:
products_df.show(10)

+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+-----------------------------+
|product_category_name|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_size_category|product_category_name_english|
+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+-----------------------------+
|           perfumaria|1e9e8ef04dbcff454...|                 40|                       287|                 1|             225|               16|               10|              14|                Small|                    perfumery|
|                artes|3aa071139cb16b67c...|                 44|    

In [0]:

dim_product = products_df

display(dim_product.limit(10))

product_category_name,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_size_category,product_category_name_english
perfumaria,1e9e8ef04dbcff4541ed26657ea517e5,40,287,1,225,16,10,14,Small,perfumery
artes,3aa071139cb16b67ca9e5dea641aaa2f,44,276,1,1000,30,18,20,Medium,art
esporte_lazer,96bd76ec8810374ed1b65e291975717f,46,250,1,154,18,9,15,Small,sports_leisure
bebes,cef67bcfe19066a932b7673e239eb23d,27,261,1,371,26,4,26,Small,baby
utilidades_domesticas,9dc1a7de274444849c219cff195d0b71,37,402,4,625,20,17,13,Medium,housewares
instrumentos_musicais,41d3672d4792049fa1779bb35283ed13,60,745,1,200,38,5,11,Small,musical_instruments
cool_stuff,732bd381ad09e530fe0a5f457d81becb,56,1272,4,18350,70,24,44,Large,cool_stuff
moveis_decoracao,2548af3e6e77a690cf3eb6368e9ab61e,56,184,2,900,40,8,40,Medium,furniture_decor
eletrodomesticos,37cc742be07708b53a98702e77a21a02,57,163,1,400,27,13,17,Small,home_appliances
brinquedos,8c92109888e8cdf9d66dc7e463025574,36,1156,1,600,17,10,12,Medium,toys


In [0]:
dim_seller = sellers_df.select(
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
)

display(dim_seller.limit(10))

seller_id,seller_city,seller_state,seller_zip_code_prefix
b3f19518fcec265b2e97af287725f981,Tanabi,SP,15170
c522be04e020c1e7b79f3acff36513d5,Santana De Parnaiba,SP,06501
a3082f442524a1be452e3189e003b361,Curitiba,SP,81560
6eeed17989b0ae47c9f11ece6f38ea90,Sao Paulo,SP,04123
8e6cc767478edae941d9bd9eb778d77a,Araguari,MG,38442
9b1050e85becf3ae99c1cf4934c9bc5d,Montes Claros,MG,39400
297d5eccd19fa9a83b2630071ff105e4,Curitiba,PR,80710
0ea22c1cfbdc755f86b9b54b39c16043,Sete Lagoas,MG,35700
0b64bcdb0784abc139af04077d49a20e,Canoas,RS,92420
9c068d10aca38e85c50202e17b4a7e88,Coxim,MS,79400


In [0]:
date_range = orders_df.select(
    to_date(
        "order_purchase_timestamp"
    ).alias("date")
).agg(
    min("date").alias("min_date"),
    max("date").alias("max_date")
).first()

assert date_range["min_date"] is not None
assert date_range["max_date"] is not None

dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            DATE '{date_range["min_date"]}',
            DATE '{date_range["max_date"]}',
            INTERVAL 1 DAY
        )) AS date
    """)
    .withColumn(
        "date_key",
        date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn(
        "day_of_week",
        dayofweek("date")
    )
    .withColumn(
        "day_type",
        when(
            dayofweek("date").isin(1, 7),
            "Weekend"
        ).otherwise("Weekday")
    )
)

display(dim_date.limit(10))

date,date_key,year,month,quarter,day_of_week,day_type
2016-09-04,20160904,2016,9,3,1,Weekend
2016-09-05,20160905,2016,9,3,2,Weekday
2016-09-06,20160906,2016,9,3,3,Weekday
2016-09-07,20160907,2016,9,3,4,Weekday
2016-09-08,20160908,2016,9,3,5,Weekday
2016-09-09,20160909,2016,9,3,6,Weekday
2016-09-10,20160910,2016,9,3,7,Weekend
2016-09-11,20160911,2016,9,3,1,Weekend
2016-09-12,20160912,2016,9,3,2,Weekday
2016-09-13,20160913,2016,9,3,3,Weekday


### Fact Tables

In [0]:
order_info = orders_df.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp"
)


In [0]:
fact_order_items = (
    order_item_df
    .join(
        order_info,
        on="order_id",
        how="left"
    )
    .withColumn(
        "date_key",
        date_format(
            to_date("order_purchase_timestamp"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "product_id",
        "seller_id",
        "date_key",
        col("price")
            .cast(DecimalType(18, 2))
            .alias("price"),
        col("freight_value")
            .cast(DecimalType(18, 2))
            .alias("freight_value")
    )
)

In [0]:
display(fact_order_items.limit(10))

order_id,order_item_id,customer_id,product_id,seller_id,date_key,price,freight_value
88498659c4bfb68808d6b49a57a6ed17,1,9c6ea4e6fbcff50e788a14bd67a29022,24ef70f010919148b022e66b05862778,a938325a4b357fd23a6a4d5bb126408e,20180301,89.00,29.02
884989e29f137d8ce42518666158b8ce,1,2ea5497e8e41a40f8f2c710b627ce684,fd76f22bf3648e0dd4295e593def7664,6d66611d7c44cc30ce351abc49a68421,20180704,39.90,13.86
8849e8100a1269ca2a5606e5a0fb6c07,1,74181092ec10162ba29946bd76ceb6b8,1cd35eaf33a642d4df4c9146837f5908,cac4c8e7b1ca6252d8f20b2fc1a2e4af,20170628,49.99,16.60
8849f02bf66502f56ce8c5173bb954df,1,905d0f8557f4baccd218a45692ec9d68,5ae2f57d379f89f995f9444cdb6d5c60,d594982fd877af63ace38ea1fca27c76,20180729,27.19,18.29
884b1394fc8888e6a877df86eb19e74c,1,7ee968b8ed74301e0cbd3e44416cad9d,437c05a395e9e47f9762e677a7068ce7,f84fa566034f5e8e880a07ec624c56af,20180319,47.65,7.39
884b42885f11cac2abce880659eb1b0b,1,5270e49915940227aa249443efb143a1,a44f1df44a1ab489c5e635885e91c9ed,f46490624488d3ff7ce78613913a7711,20180814,32.90,12.53
884c8bf4713eff2d1830882d96a8f90e,1,f06a932d9fefe343141d0570ca5d5721,b38b25d838ae0b8385e8cc68b9017644,fad44952713764836814be105382aee5,20180515,130.00,7.93
884cb4ce4a06390d35206de06f785d13,1,d19813c8754cc82d0add67df8bd8a38c,0a8f26369822d83b82d0d3e87908fbd4,f262cbc1c910c83959f849465454ddd3,20180403,16.99,7.39
884dd22244240995c8a18169c32be54d,1,746704bb5f6ca629195b6a5195014c50,0aabfb375647d9738ad0f7b4ea3653b1,37515688008a7a40ac93e3b2e4ab203f,20170928,19.90,15.10
884ddcd0965c8f4d41006f07efa3c32b,1,f34cd6c38354857985ae92056f6c2db8,2d4d78eecba060a7127c5f6a21cdb0c7,a673821011d0cec28146ea42f5ab767f,20171120,99.90,13.72


In [0]:
order_totals = (
    order_item_df
    .groupBy("order_id")
    .agg(
        sum(
            col("price").cast(DecimalType(18, 2))
        ).alias("order_revenue"),
        sum(
            col("freight_value")
             .cast(DecimalType(18, 2))
        ).alias("order_freight"),
        count("*").alias("item_count")
    )
)

fact_orders = (
    orders_df
    .join(
        order_totals,
        on="order_id",
        how="left"
    )
    .withColumn(
        "date_key",
        date_format(
            to_date("order_purchase_timestamp"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "delivery_days",
        datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "is_delivered",
        when(
            col("order_status") == "delivered", 1
        ).otherwise(0)
    )
    .withColumn(
        "is_canceled",
        when(
            col("order_status") == "canceled", 1
        ).otherwise(0)
    )
    .select(
        "order_id",
        "customer_id",
        "date_key",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "is_delivered",
        "is_canceled",
        "item_count",
        "order_revenue",
        "order_freight"
    )
)

display(fact_orders.limit(10))

order_id,customer_id,date_key,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_delivered,is_canceled,item_count,order_revenue,order_freight
bcc07c3639d84c50bb1786b8bcc66116,756f09dbf94f877b9acce94031f05714,20180429,delivered,2018-04-29,2018-05-04T15:04:38Z,2018-05-14T00:00:00Z,5,1,0,1,133.00,11.56
b1fcaac6e8a89bb18db51bcb62c4a4e6,1b60bc9ccf0b19e5b4cf1e59c76afc5f,20180514,delivered,2018-05-14,2018-06-01T11:08:39Z,2018-06-06T00:00:00Z,18,1,0,1,265.00,38.77
c3e544a509d5603803d4f6eaedcaf0b8,67ccb8621f5cb3f53b797fe9af6143c6,20170807,delivered,2017-08-07,2017-08-11T13:12:42Z,2017-08-29T00:00:00Z,4,1,0,1,59.90,15.18
e6a1dffced3033ae0e4838aece8aabc6,4e7bafa1bcb313b042aa22868c81aa8c,20180507,delivered,2018-05-07,2018-05-11T19:12:03Z,2018-05-24T00:00:00Z,4,1,0,1,59.99,13.78
acf93693a6780a86fb735f6743d5bcf7,f0941f0a360451bb222634e610f7ad08,20170730,delivered,2017-07-30,2017-08-07T17:27:06Z,2017-08-24T00:00:00Z,8,1,0,1,135.90,17.20
eae6b90fc5f39d2d76ea4edb84ade594,40aa5aa3e0a9d5cbcec2281f2927b576,20180508,delivered,2018-05-08,2018-05-16T19:04:28Z,2018-05-18T00:00:00Z,8,1,0,1,50.90,14.29
9a602b72c06820975aaa136e4fb793b5,f0529c2f06ee170b6dd5d0a3ddb7a55b,20180318,delivered,2018-03-18,2018-03-22T19:19:39Z,2018-04-13T00:00:00Z,4,1,0,1,17.00,7.87
a76643614c2542c0a773b03b5b61b2c0,d557ba0b467c10c23bc9542050ba64a2,20170306,delivered,2017-03-06,2017-03-20T11:11:05Z,2017-04-03T00:00:00Z,14,1,0,1,129.99,71.74
b52314e04baaecb498e1446be2dcabee,cb7e2ee52c84733eedbaac0f2527fe3a,20180304,delivered,2018-03-04,2018-04-14T10:21:59Z,2018-03-29T00:00:00Z,41,1,0,1,199.70,38.95
1fb40b224a9509697bacc23d3e160e4c,0d993384667c2d9151c18374570d286f,20170608,delivered,2017-06-08,2017-06-19T21:29:38Z,2017-07-10T00:00:00Z,11,1,0,1,44.90,16.11


### Data Validation

In [0]:
# Fact row counts
assert fact_order_items.count() == order_item_df.count()
assert fact_orders.count() == orders_df.count()

# Fact key uniqueness
check_unique(
    fact_order_items,
    ["order_id", "order_item_id"],
    "fact_order_items"
)

check_unique(
    fact_orders,
    ["order_id"],
    "fact_orders"
)

# Look for missing parent orders
missing_orders = (
    fact_order_items
    .filter(col("customer_id").isNull())
    .count()
)

print("Items without parent order:", missing_orders)

# Compare revenue to Silver order items
silver_total = order_item_df.agg(
    sum(
        col("price").cast(DecimalType(18, 2))
    )
).first()[0]

gold_total = fact_order_items.agg(
    sum("price")
).first()[0]

assert silver_total == gold_total

print("Revenue validation passed")

Items without parent order: 0
Revenue validation passed


In [0]:
# So here we the list of all dimensional and fact tables that we are going to save as parquet files in ADLS GOLD Layer


# 1. Customer Dimensional Table
dim_customer.write.mode('overwrite').parquet(adls_gold_path + 'dim_customer/')

# 2. Product Dimensional Table
dim_product.write.mode('overwrite').parquet(adls_gold_path + 'dim_product/')

# 3. Seller Dimensional Table
dim_seller.write.mode('overwrite').parquet(adls_gold_path + 'dim_seller/')

# 4. Date Dimensional Table
dim_date.write.mode('overwrite').parquet(adls_gold_path + 'dim_date/')

# 5. Fact Orders
fact_orders.write.mode('overwrite').parquet(adls_gold_path + 'fact_orders/')

# 6. Fact Order Items
fact_order_items.write.mode('overwrite').parquet(adls_gold_path + 'fact_order_items/')


print("All Dimensional and Fact Dataframes saved as Parquet files successfully!")

All Dimensional and Fact Dataframes saved as Parquet files successfully!
